In [1]:
import pycaret
print(pycaret.__version__) 

3.3.2


In [2]:
import numpy as np
SEED = 7
np.random.seed(SEED) 

In [3]:
import warnings
warnings.filterwarnings("ignore") 

In [4]:
import pandas as pd
from pycaret.classification import *

# Load your preprocessed dataset
df = pd.read_csv(r"C:\Users\manis\Downloads\preprocessed_multiple_class.csv")

# Target column
target = 'attack type'

# Show quick overview
print(df.shape)
df.head() 

(35000, 36)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC27,PC28,PC29,PC30,PC31,PC32,PC33,PC34,PC35,attack type
0,-1.844530,0.003013,-0.077057,-0.214017,-0.604202,0.785226,0.088003,0.488782,-1.089147,-0.266907,...,-0.399339,0.442995,0.380217,0.086113,0.398695,-0.038791,-0.000418,-0.035242,-0.071810,Bot
1,-2.114291,-0.053045,0.379540,-0.046920,1.242315,0.119263,0.000331,-0.628133,-0.882525,2.033132,...,0.033110,-1.786123,0.818943,-0.538143,-0.264298,0.011855,-0.000585,0.031274,0.022991,Web Attack
2,-1.842625,0.002847,-0.075887,-0.213708,-0.604181,0.785390,0.087978,0.487524,-1.089749,-0.267633,...,-0.400734,0.442891,0.379991,0.086025,0.398904,-0.038814,-0.000419,-0.034028,-0.072332,Bot
3,-1.478224,-0.010469,0.011352,-0.011329,-0.400250,1.592338,0.161982,0.952709,-2.091688,-0.096761,...,-0.242185,-0.208548,0.288761,0.077288,0.315320,-0.026143,-0.000367,0.311215,-0.128024,Web Attack
4,-1.902913,0.026170,-0.273082,-0.080753,-0.567086,1.085976,0.120778,0.802664,-1.394595,-0.442245,...,-0.094592,0.409424,0.065545,0.266979,0.509591,-0.036566,-0.001091,0.010120,-0.101205,Port Scan


In [5]:
# Setup PyCaret
s = setup(
    data=df,
    target=target,
    session_id=42,
    log_experiment=False,
    use_gpu=False
) 

,Description,Value
0,Session id,42
1,Target,attack type
2,Target type,Multiclass
3,Target mapping,"BENIGN: 0, Bot: 1, Brute Force: 2, DDoS: 3, DoS: 4, Port Scan: 5, Web Attack: 6"
4,Original data shape,"(35000, 36)"
5,Transformed data shape,"(35000, 36)"
6,Transformed train set shape,"(24500, 36)"
7,Transformed test set shape,"(10500, 36)"
8,Numeric features,35
9,Preprocess,True


In [6]:
best_model = compare_models(sort='F1', n_select=1)
print(best_model) 

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.9928,0.9998,0.9928,0.9928,0.9928,0.9916,0.9916,2.7420
xgboost,Extreme Gradient Boosting,0.9923,0.9998,0.9923,0.9923,0.9923,0.9910,0.9910,2.4380
et,Extra Trees Classifier,0.9921,0.9996,0.9921,0.9921,0.9921,0.9908,0.9908,0.3690
rf,Random Forest Classifier,0.9914,0.9995,0.9914,0.9914,0.9914,0.9900,0.9900,1.8770
catboost,CatBoost Classifier,0.9914,0.9997,0.9914,0.9914,0.9914,0.9900,0.9900,28.7580
gbc,Gradient Boosting Classifier,0.9884,0.0000,0.9884,0.9884,0.9884,0.9864,0.9864,46.6790
dt,Decision Tree Classifier,0.9856,0.9916,0.9856,0.9856,0.9855,0.9831,0.9832,0.3770
qda,Quadratic Discriminant Analysis,0.9842,0.0000,0.9842,0.9844,0.9842,0.9815,0.9815,0.0890
knn,K Neighbors Classifier,0.9818,0.9961,0.9818,0.9819,0.9817,0.9788,0.9789,1.4900
lr,Logistic Regression,0.9532,0.0000,0.9532,0.9548,0.9524,0.9454,0.9459,2.0410


LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
               importance_type='split', learning_rate=0.1, max_depth=-1,
               min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
               n_estimators=100, n_jobs=-1, num_leaves=31, objective=None,
               random_state=42, reg_alpha=0.0, reg_lambda=0.0, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)


In [7]:
# Evaluate performance visually
evaluate_model(best_model) 

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [8]:
multiclass_models = [
    'lightgbm',
    'xgboost',
    'et',
    'rf',
    'catboost'
] 

In [9]:
from pycaret.classification import *
import pandas as pd

def get_fold_scores(model_list, metric="F1", fold=5):

    scores = {}

    for model_name in model_list:

        print(f"Training {model_name}...")

        create_model(model_name, fold=fold)

        results = pull()

        # Remove Mean and Std rows
        fold_scores = results.iloc[:-2][metric].astype(float).tolist()

        scores[model_name] = fold_scores

    return scores

In [10]:
multiclass_scores = get_fold_scores(
    multiclass_models,
    metric="F1",
    fold=5
)

Training lightgbm...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9914,0.9997,0.9914,0.9914,0.9914,0.9900,0.9900
1,0.9920,0.9998,0.9920,0.9921,0.9920,0.9907,0.9907
2,0.9910,0.9996,0.9910,0.9910,0.9910,0.9895,0.9895
3,0.9916,0.9999,0.9916,0.9916,0.9916,0.9902,0.9902
4,0.9922,0.9998,0.9922,0.9922,0.9922,0.9910,0.9910
Mean,0.9917,0.9998,0.9917,0.9917,0.9917,0.9903,0.9903
Std,0.0004,0.0001,0.0004,0.0004,0.0004,0.0005,0.0005


Training xgboost...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9914,0.9998,0.9914,0.9914,0.9914,0.9900,0.9900
1,0.9912,0.9998,0.9912,0.9913,0.9912,0.9898,0.9898
2,0.9908,0.9996,0.9908,0.9908,0.9908,0.9893,0.9893
3,0.9918,0.9998,0.9918,0.9918,0.9918,0.9905,0.9905
4,0.9935,0.9997,0.9935,0.9935,0.9935,0.9924,0.9924
Mean,0.9918,0.9997,0.9918,0.9918,0.9917,0.9904,0.9904
Std,0.0009,0.0001,0.0009,0.0009,0.0009,0.0011,0.0011


Training et...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9916,0.9996,0.9916,0.9916,0.9916,0.9902,0.9902
1,0.9910,0.9995,0.9910,0.9910,0.9910,0.9895,0.9895
2,0.9906,0.9994,0.9906,0.9906,0.9906,0.9890,0.9891
3,0.9902,0.9995,0.9902,0.9902,0.9902,0.9886,0.9886
4,0.9922,0.9995,0.9922,0.9923,0.9922,0.9910,0.9910
Mean,0.9911,0.9995,0.9911,0.9911,0.9911,0.9897,0.9897
Std,0.0007,0.0001,0.0007,0.0007,0.0007,0.0008,0.0008


Training rf...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9908,0.9997,0.9908,0.9908,0.9908,0.9893,0.9893
1,0.9912,0.9995,0.9912,0.9913,0.9912,0.9898,0.9898
2,0.9898,0.9994,0.9898,0.9898,0.9898,0.9881,0.9881
3,0.9906,0.9997,0.9906,0.9906,0.9906,0.9890,0.9891
4,0.9924,0.9994,0.9924,0.9925,0.9924,0.9912,0.9912
Mean,0.9910,0.9995,0.9910,0.9910,0.9910,0.9895,0.9895
Std,0.0009,0.0001,0.0009,0.0009,0.0009,0.0010,0.0010


Training catboost...


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9904,0.9998,0.9904,0.9904,0.9904,0.9888,0.9888
1,0.9908,0.9998,0.9908,0.9909,0.9908,0.9893,0.9893
2,0.9902,0.9995,0.9902,0.9902,0.9902,0.9886,0.9886
3,0.9914,0.9998,0.9914,0.9914,0.9914,0.9900,0.9900
4,0.9914,0.9997,0.9914,0.9914,0.9914,0.9900,0.9900
Mean,0.9909,0.9997,0.9909,0.9909,0.9908,0.9893,0.9893
Std,0.0005,0.0001,0.0005,0.0005,0.0005,0.0006,0.0006


In [11]:
from itertools import combinations
from scipy.stats import ttest_rel, wilcoxon
import pandas as pd

def pairwise_tests(scores):

    rows = []

    for m1, m2 in combinations(scores.keys(), 2):

        t_stat, t_p = ttest_rel(scores[m1], scores[m2])

        w_stat, w_p = wilcoxon(scores[m1], scores[m2])

        rows.append({
            "Model 1": m1,
            "Model 2": m2,
            "t-statistic": round(t_stat,4),
            "t p-value": round(t_p,6),
            "Wilcoxon Statistic": round(w_stat,4),
            "Wilcoxon p-value": round(w_p,6)
        })

    return pd.DataFrame(rows) 

In [12]:
multiclass_stats = pairwise_tests(multiclass_scores)

multiclass_stats

,Model 1,Model 2,t-statistic,t p-value,Wilcoxon Statistic,Wilcoxon p-value
0,lightgbm,xgboost,-0.2911,0.785437,4.5,0.853923
1,lightgbm,et,1.7295,0.158776,1.0,0.144127
2,lightgbm,rf,2.8139,0.048128,1.0,0.125000
3,lightgbm,catboost,4.7809,0.008770,0.0,0.062500
4,xgboost,et,1.7722,0.151060,3.0,0.312500
5,xgboost,rf,3.5455,0.023896,0.0,0.067889
6,xgboost,catboost,2.8180,0.047925,0.0,0.062500
7,et,rf,0.6065,0.576933,6.0,0.812500
8,et,catboost,0.6864,0.530178,4.0,0.437500
9,rf,catboost,0.3750,0.726697,5.5,0.812500


In [13]:
!pip install autogluon

  Using cached hyperopt-0.2.7-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached torch-2.9.1-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
     ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
     --- ------------------------------------ 0.3/2.9 MB ? eta -:--:--
     --- ------------------------------------ 0.3/2.9 MB ? eta -:--:--
     --- ------------

  DEPRECATION: Building 'nvidia-ml-py3' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'nvidia-ml-py3'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'antlr4-python3-runtime' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'antlr4-python3-runtime'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'seqeval' using th

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.metrics import accuracy_score, f1_score

predictor = TabularPredictor(
    label=target,
    eval_metric="f1"
).fit(
    train_data=train_df,
    presets="best_quality",
    time_limit=3600
)

pred = predictor.predict(test_df) 

print("Accuracy:", accuracy_score(test_df[target], pred))
print("F1:", f1_score(test_df[target], pred))